# IIT_DroneLearning — Evader RL 학습 (Colab Pro+)

**담당**: 이재왕 (work/evader)  
**현재 단계**: Stage0-A — 순수 목표 도달 선행 학습 (`_goalOnlyMode = true`)

## 전체 흐름
```
Stage0-A (이 노트북) → goal_reach_rate ≥ 70% 확인
       ↓ --initialize-from 으로 가중치 이전
Stage0-B (같은 노트북, GOAL_ONLY=False) → M1 DoD 달성
       ↓
Stage1  → 장애물 + 힌트 제거
```

## 실행 전 필수 체크리스트

### Unity 씬 설정 (로컬에서 한 번)
DroneTest 씬을 열고 Evader 드론 오브젝트에 아래 컴포넌트를 추가/설정한다.

| 컴포넌트 | 설정값 |
|---|---|
| `DronePhysics` | ThrustForce=10, MaxSpeed=8, MaxAltitude=30 (기본값) |
| `EvaderReward` | goalShapingCoeff=0.02, survivalPerStep=0.001, timePenalty=-0.001 |
| `EvaderAgent` | **_goalOnlyMode=✅**, _goalTransform=GoalZone, _pursuerTransform=**(비움)**, _dronePhysics 연결 |
| `BehaviorParameters` | Behavior Name=**EvaderAgent**, Space Type=Continuous, Actions=**4**, Observations=**18** |
| `DecisionRequester` | Decision Period=**5** (50Hz FixedUpdate → 10Hz 결정) |
| `RayPerceptionSensor` | Sensor Name=RaySensor (Sensor 담당 배민우 설정) |

> ⚠️ BehaviorParameters의 Behavior Name이 **EvaderAgent** 여야 YAML과 매핑됩니다.

### Colab 준비
- [ ] GPU 런타임 활성화 (메뉴 > 런타임 > 런타임 유형 변경 > T4 GPU)
- [ ] Google Drive에 Unity Linux Headless 빌드 업로드 완료
- [ ] `DRIVE_BASE` 경로를 실제 Drive 경로로 수정

---
## 0. Colab 연결 유지 (실행 후 방치할 때)

In [ ]:
# Colab 자동 연결 해제 방지 (브라우저 콘솔에서 실행)
# 아래 코드를 브라우저 개발자 도구 Console에 붙여넣기:
# function ClickConnect(){console.log('Colab 연결 유지'); document.querySelector('#top-toolbar > colab-connect-button').shadowRoot.querySelector('#connect').click();} setInterval(ClickConnect, 60000);
print('위 JavaScript를 브라우저 Console에 붙여넣으면 자동 연결 해제를 방지할 수 있습니다.')

---
## 1. 환경 설치

In [ ]:
import sys

# 1. pettingzoo==1.15.0 은 PyPI에 없음 → 호환 버전(1.22.3)으로 선설치
#    (mlagents-envs release_23 이 pettingzoo==1.15.0 을 pin하지만 API 호환됨)
!{sys.executable} -m pip install -q "pettingzoo==1.22.3"

# 2. mlagents-envs: --no-deps 로 pettingzoo 버전 핀 우회
!{sys.executable} -m pip install -q --no-deps \
    "mlagents-envs @ git+https://github.com/Unity-Technologies/ml-agents.git@release_23#subdirectory=ml-agents-envs"

# 3. mlagents: --no-deps 로 mlagents_envs==1.2.0.dev0 (PyPI 없음) 우회
!{sys.executable} -m pip install -q --no-deps \
    "mlagents @ git+https://github.com/Unity-Technologies/ml-agents.git@release_23#subdirectory=ml-agents"

# 4. 나머지 의존성 호환 버전으로 수동 설치
#    grpcio>1.53 / protobuf>=4 / numpy>=2 는 mlagents release_23 과 충돌
!{sys.executable} -m pip install -q \
    "grpcio>=1.11.0,<=1.53.2" \
    "protobuf>=3.20.0,<4.0" \
    "numpy>=1.21.0,<2.0" \
    "h5py>=2.9.0" \
    "Pillow>=4.2.1" \
    "PyYAML>=5.1" \
    "cattrs>=1.1.0" \
    "attrs>=21.0.0"

# 5. PyTorch 및 학습 도구
!{sys.executable} -m pip install -q "torch>=2.0.0,<3.0.0" tensorboard onnx

# 설치 확인
import importlib; importlib.invalidate_caches()
import mlagents, torch
print(f'mlagents : {mlagents.__version__}')
print(f'torch    : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
print(f'Python   : {sys.version}')

---
## 2. Google Drive 마운트 및 경로/실험 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ── 경로 설정 ────────────────────────────────────────────────────
DRIVE_BASE     = '/content/drive/MyDrive/IIT_DroneLearning'  # ← 실제 경로로 수정
REPO_PATH      = '/content/IIT_DroneLearning'
BUILD_PATH     = f'{DRIVE_BASE}/builds/EvaderEnv'
LOG_DIR        = f'{DRIVE_BASE}/runs'
CHECKPOINT_DIR = f'{DRIVE_BASE}/checkpoints'

for d in [LOG_DIR, CHECKPOINT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── 실험 설정 ────────────────────────────────────────────────────
# Stage0-A: 순수 목표 도달 (goalOnlyMode = true, pursuerTransform 없음)
# Stage0-B: Pursuer GT 힌트 추가 (goalOnlyMode = false) ← 수렴 후 전환
GOAL_ONLY  = True    # Stage0-A=True / Stage0-B=False
STAGE      = 0
CONFIG_DATE = '20260315'
SEED       = 42
NUM_ENVS   = 4       # Colab Pro+ T4 기준
INIT_FROM  = None    # Stage0-B 전환 시: 'evader_s0a_goalonly_seed42'

phase      = 'goalonly' if GOAL_ONLY else 'withpursuer'
RUN_ID     = f'evader_s{STAGE}a_{phase}_seed{SEED}' if GOAL_ONLY \
             else f'evader_s{STAGE}b_{phase}_seed{SEED}'

print(f'Run ID  : {RUN_ID}')
print(f'Log dir : {LOG_DIR}')
print(f'Phase   : {"Stage0-A (goal only)" if GOAL_ONLY else "Stage0-B (with pursuer)"}')

---
## 3. 레포 클론 및 Config 확인

In [ ]:
# 레포 클론 (이미 존재하면 pull)
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/alpha7179/IIT_DroneLearning.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git fetch origin work/evader && git reset --hard origin/work/evader

!cd {REPO_PATH} && git checkout work/evader && git log --oneline -3

# Config 경로 — 날짜판 우선, 없으면 template 폴백
config_candidates = [
    f'{REPO_PATH}/python/config/evader_s{STAGE}_{CONFIG_DATE}_base.yaml',
    f'{REPO_PATH}/python/config/evader_s{STAGE}_template.yaml',
]
config_path = next((c for c in config_candidates if os.path.exists(c)), None)
assert config_path, f'Config not found: {config_candidates}'

print(f'\nConfig: {config_path}')
print('=' * 60)
!cat {config_path}

---
## 4. Unity Linux Headless 빌드 준비

### 로컬 Unity에서 한 번만 수행
```
1. File > Build Settings
2. Platform: Linux (x86_64)  ← 플랫폼 미설치 시 Install 필요
3. ✅ Server Build (헤드리스 모드)
4. Build → 파일명: EvaderEnv
5. 빌드 폴더(EvaderEnv/)를 Drive/IIT_DroneLearning/builds/ 에 업로드
```

### 씬 설정 확인 (빌드 전 필수)
DroneTest 씬에서:
- Evader 드론 오브젝트에 **EvaderAgent, DronePhysics, EvaderReward** 컴포넌트 붙이기
- BehaviorParameters: Behavior Name = `EvaderAgent`, Continuous Actions = 4
- Vector Observation Size = **18** (RayPerception은 별도 컴포넌트)
- DecisionRequester: Decision Period = **5**
- EvaderAgent: **_goalOnlyMode = ✅**

In [ ]:
build_exe = f'{BUILD_PATH}/EvaderEnv.x86_64'

if os.path.exists(build_exe):
    !chmod +x {build_exe}
    print(f'✅ Build found: {build_exe}')
    BUILD_READY = True
else:
    print(f'❌ Build NOT found: {build_exe}')
    print('Unity Linux Headless 빌드를 Drive에 업로드하세요.')
    BUILD_READY = False

---
## 5. 학습 실행

In [ ]:
import subprocess

assert BUILD_READY, '빌드 파일이 없습니다. 4번 셀을 먼저 확인하세요.'

cmd = [
    'mlagents-learn', config_path,
    f'--run-id={RUN_ID}',
    f'--results-dir={LOG_DIR}',
    f'--env={build_exe}',
    f'--num-envs={NUM_ENVS}',
    '--no-graphics',   # 헤드리스 빌드에 필수
    '--force',
]

if INIT_FROM:
    init_path = str(Path(LOG_DIR) / INIT_FROM)
    cmd += [f'--initialize-from={init_path}']
    print(f'Warm-start from: {init_path}')

print('실행 커맨드:')
print(' '.join(cmd))
print()

# 학습 시작 (블로킹 — Colab 런타임이 살아있는 동안 학습 진행)
result = subprocess.run(cmd, cwd=REPO_PATH)
print(f'\n학습 종료. Exit code: {result.returncode}')

---
## 6. TensorBoard 모니터링 (학습 중 병렬 실행)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

# 확인해야 할 지표:
# Environment/Episode Length     : 너무 짧으면 crash/즉시종료 의심
# Environment/Cumulative Reward  : 상승 추세여야 정상
# Policy/Entropy                 : 초반 높고 서서히 감소
# Losses/Policy Loss             : 0 근방에서 안정화
#
# Stage0-A 수렴 기준:
#   Cumulative Reward > 0.5 이상 안정
#   Episode Length가 줄어들고 있음 (더 빨리 Goal 도달)
#   → eval로 goal_reach_rate >= 70% 확인 후 Stage0-B 전환

---
## 7. 체크포인트 Drive 저장 (학습 완료 후)

In [ ]:
import shutil

src = Path(LOG_DIR) / RUN_ID
dst = Path(CHECKPOINT_DIR) / RUN_ID

if src.exists():
    shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
    print(f'✅ Saved: {dst}')

    onnx_files = list(dst.glob('**/*.onnx'))
    if onnx_files:
        print(f'ONNX models ({len(onnx_files)}개):')
        for f in onnx_files:
            print(f'  {f}')
    else:
        print('ONNX 파일 없음 — max_steps 미도달 또는 export 필요')
else:
    print(f'❌ Run directory not found: {src}')
    print('학습이 정상 완료되지 않았거나 LOG_DIR 경로를 확인하세요.')

---
## 8. Stage0-A → Stage0-B 전환 (수렴 확인 후)

```python
# 위 2번 셀에서 아래와 같이 변경 후 5번 셀 재실행:
GOAL_ONLY = False
INIT_FROM = 'evader_s0a_goalonly_seed42'  # Stage0-A run-id
```

Unity Inspector에서 변경:
- `EvaderAgent._goalOnlyMode` = **false** (체크 해제)
- `EvaderAgent._pursuerTransform` = ScriptedPursuer 오브젝트 연결

그 후 Linux 빌드 재수행 후 위 5번 셀 재실행.

---
## 9. 실험 결과 기록

학습 완료 후 로컬에서 `Docs/EXPERIMENTS.md`에 기록:
```bash
# Docs/EXPERIMENTS.md 테이블에 한 줄 추가:
# | 2026-03-15 | <commit> | evader_s0_20260315_base.yaml | evader_s0a_goalonly_seed42 | 42 | -% | ??% | -% | -% | Stage0-A 첫 수렴 확인 |
git add Docs/EXPERIMENTS.md
git commit -m '[Docs] Stage0-A 실험 결과 기록'
git push origin work/evader
```